# BM25 Gets Its Revenge: When the LLM Should Compile the Query, Not Run the Search

Companion notebook for the article. It rebuilds the core intuition of Meta's **SIRA** ([arXiv:2605.06647](https://arxiv.org/abs/2605.06647)) on a small test corpus and compares five retrieval systems:

1. **BM25** - vanilla lexical retrieval
2. **Dense** - `all-MiniLM-L6-v2` embeddings, cosine similarity
3. **Naive expansion** - BM25 over the query plus corpus-blind synonyms
4. **Corpus-aware expansion** - SIRA-style query-side only: candidate terms filtered by document frequency, then one weighted BM25 call (does not include offline document enrichment or reranking)
5. **Hybrid** - reciprocal rank fusion of (1) and (2)

Everything runs locally and deterministically. The `candidates` lists were **hand-authored** (not model-generated) to simulate well-prompted LLM output; an optional cell at the end regenerates them with a live model.

Requirements: `pip install -r requirements.txt`

Scope: this notebook implements query-side DF-filtered expansion only. Full SIRA also enriches documents offline at index time.

In [ ]:
# %pip install -r requirements.txt
import re
import numpy as np
from rank_bm25 import BM25Okapi

## 1. A small corpus built to break retrievers

36 documents from a fictional SaaS company's knowledge base, deliberately seeded with realistic failure modes: exact identifiers (`ERR-4032`, `v2.14.0`) that embeddings blur together, vocabulary mismatch (users say *slow*, docs say *latency degradation*), and confuser documents that share generic vocabulary with the queries.

Each query has one gold document and two frozen term lists: `naive` (corpus-blind synonyms) and `candidates` (hand-authored guesses, validated by document frequency).

In [ ]:
DOCS = {
    # Billing / payments
    "pay-err-4032": (
        "Error ERR-4032: card declined due to AVS mismatch. ERR-4032 is raised "
        "by the billing gateway when the address verification system rejects the "
        "cardholder address. Ask the customer to confirm their billing address "
        "and postal code, then retry the charge. ERR-4032 is permanent for the "
        "attempt; retrying without changes will fail again."
    ),
    "pay-err-4031": (
        "Error ERR-4031: card declined due to insufficient funds. The billing "
        "gateway returns ERR-4031 when the issuing bank rejects the charge. The "
        "customer should use a different card or retry after funds are available."
    ),
    "pay-general": (
        "Troubleshooting failed payments. When a payment fails, check the gateway "
        "dashboard for the decline code, verify the customer's card is not "
        "expired, and confirm the account is in good standing. Most payment "
        "errors resolve after the customer updates their card details."
    ),
    "pay-retry-policy": (
        "Dunning and retry policy. Failed subscription charges are retried on a "
        "3-5-7 day schedule. After the final retry the account is downgraded to "
        "the free plan and the user receives a payment failure email."
    ),
    "vat-germany": (
        "Invoice tax bug VAT-1203: wrong VAT rate applied to German customers. "
        "Invoices generated between May 2 and May 9 applied the Austrian 20% "
        "rate instead of the German 19% rate for customers with a DE country "
        "code. Affected invoices must be regenerated with a corrected tax line."
    ),
    "billing-cycle": (
        "How billing cycles work. Subscriptions renew on the anniversary of the "
        "signup date. Mid-cycle plan changes are prorated. Invoices are emailed "
        "to the account owner and available under Settings, then Billing."
    ),

    # Auth / accounts
    "twofa-reset": (
        "Resetting two-factor authentication for a locked-out user. If a user "
        "lost their TOTP device, verify identity with two recovery signals "
        "(billing email plus last-4 of card on file), then clear the TOTP seed "
        "from the admin console. The user re-enrolls at next sign-in. Never "
        "disable enforcement tenant-wide to unblock a single user."
    ),
    "account-security": (
        "Account security best practices. Encourage users to enable two-factor "
        "authentication, use a password manager, and review active sessions. "
        "Suspicious login attempts trigger an email alert and may temporarily "
        "lock the account until the user verifies access."
    ),
    "account-deactivate": (
        "Deactivating and reactivating accounts. Admins can deactivate a user "
        "account from the admin console. Deactivation removes access but keeps "
        "user data intact. Reactivating restores the account exactly as it was. "
        "Deactivation is not deletion; no information is removed."
    ),
    "gdpr-erasure": (
        "Handling GDPR right-to-erasure requests. Under GDPR Article 17 a "
        "customer may request erasure of their personal data. File a privacy "
        "ticket; the privacy job purges profile records, support transcripts and "
        "analytics identifiers within 30 days, then emails a completion receipt."
    ),
    "password-policy": (
        "Password policy. Passwords require 12 characters minimum. Users can "
        "reset a forgotten password from the sign-in page; reset links expire "
        "after 30 minutes. Repeated failed attempts lock the account for 15 "
        "minutes."
    ),
    "session-mgmt": (
        "Session management. Sessions expire after 14 days of inactivity. Users "
        "can review and revoke active sessions under Settings. Revoking a "
        "session signs that device out immediately."
    ),

    # Pricing / plans
    "pricing-tiers": (
        "Plan comparison. Starter includes core features for small teams. "
        "Business adds audit logs, priority support and usage analytics. "
        "Enterprise adds SAML single sign-on, SCIM provisioning, a dedicated "
        "success manager and a 99.95% uptime SLA. Single sign-on is available "
        "on the Enterprise tier only."
    ),
    "plan-limits": (
        "Plan limits and quotas. Starter allows 5 seats and 10,000 API calls "
        "per day. Business allows 50 seats and 100,000 API calls. Enterprise "
        "limits are negotiated per contract. Exceeding a quota returns HTTP 429."
    ),
    "discounts": (
        "Discounts and nonprofit pricing. Annual billing saves 20% on any plan. "
        "Registered nonprofits and educational institutions qualify for an "
        "additional 30% discount on Business and Enterprise plans."
    ),

    # Releases / changelogs
    "changelog-2-13": (
        "Release notes v2.13.0. Added bulk export to CSV, redesigned the "
        "notification center, and fixed a race condition in the import worker. "
        "Deprecated the legacy webhooks API."
    ),
    "changelog-2-14": (
        "Release notes v2.14.0. Introduced the new permissions model with "
        "custom roles, added dark mode, and upgraded the dashboard charting "
        "library. Known issue: report pages may load slowly for large "
        "workspaces; fixed in v2.14.1."
    ),
    "changelog-2-15": (
        "Release notes v2.15.0. Shipped the audit log search API, improved "
        "mobile responsiveness, and reduced cold-start time for serverless "
        "functions. Removed the deprecated legacy webhooks API."
    ),
    "perf-regression": (
        "Incident retro: latency degradation after the v2.14.0 rollout. The new "
        "permissions model introduced an N+1 query pattern on the dashboard "
        "endpoint, raising p95 response times from 180ms to 2.4s for large "
        "workspaces. Mitigated by eager-loading role assignments and adding a "
        "covering index. Permanent fix shipped in v2.14.1."
    ),

    # Infra / ops
    "crashloop": (
        "Debugging CrashLoopBackOff in the production cluster. A container that "
        "exits shortly after start enters CrashLoopBackOff with exponential "
        "backoff. Check kubectl logs --previous for the crash output, verify "
        "liveness probe timeouts, and confirm required environment variables "
        "and secrets are mounted. Most cases trace to a failed migration on "
        "boot or an OOMKilled container with too low a memory limit."
    ),
    "k8s-deploy": (
        "Deploying services to Kubernetes. Services deploy via the CI pipeline "
        "which builds the image, pushes to the registry and applies the "
        "manifests. Rollouts use a 25% surge strategy. Use kubectl rollout "
        "status to watch a deployment complete."
    ),
    "terraform-lock": (
        "Terraform state lock errors. If terraform apply hangs or fails with a "
        "state lock error, a previous run likely crashed while holding the "
        "lock. Inspect the lock with the DynamoDB console, confirm no apply is "
        "actually running, then release it with terraform force-unlock LOCK_ID. "
        "Never force-unlock while a teammate's apply is in progress."
    ),
    "ci-pipeline": (
        "CI pipeline overview. Every merge to main triggers build, test and "
        "deploy stages. Flaky tests can be retried from the pipeline view. A "
        "red deploy stage pages the on-call engineer."
    ),
    "redis-pool": (
        "Worker timeouts caused by Redis connection pool exhaustion. Background "
        "workers opening one Redis connection per job exhaust the pool under "
        "load, and new jobs block until they time out. Configure a shared "
        "connection pool with max_connections sized to worker concurrency, and "
        "set socket_timeout so stuck connections fail fast."
    ),
    "redis-cache": (
        "Caching strategy. We cache rendered dashboard fragments in Redis with "
        "a 5 minute TTL. Cache keys include the workspace id and role hash. "
        "Invalidation happens on permission changes via a pub/sub channel."
    ),
    "pg-upgrade": (
        "Zero-downtime PostgreSQL major version upgrade. Use logical "
        "replication: provision a replica on the new version, replicate until "
        "lag is zero, pause writes briefly, switch the application connection "
        "string, then decommission the old primary. Rehearse the cutover in "
        "staging and keep the old primary for fast rollback."
    ),
    "pg-tuning": (
        "PostgreSQL tuning basics. Watch for sequential scans on large tables, "
        "set shared_buffers to roughly 25% of memory, and enable "
        "log_min_duration_statement to find slow queries."
    ),
    "incident-process": (
        "Incident response process. Declare an incident in the alerts channel, "
        "assign an incident commander, post status updates every 30 minutes, "
        "and write a retro within 5 working days. Sev1 means customer-facing "
        "outage; Sev2 means degraded service."
    ),

    # Integrations / API
    "webhook-rotate": (
        "Rotating a webhook signing secret. Each webhook endpoint has a signing "
        "secret used to compute the X-Signature header. To rotate without "
        "dropping events, add a second secret in the dashboard, deploy receiver "
        "code that accepts both, then revoke the old secret after 24 hours."
    ),
    "webhook-debug": (
        "Debugging webhook deliveries. The dashboard shows delivery attempts, "
        "response codes and payloads for each endpoint. Failed deliveries are "
        "retried with exponential backoff for up to 72 hours."
    ),
    "api-keys": (
        "API key management. Create scoped API keys from the developer "
        "settings. Keys are shown once at creation. Revoking a key invalidates "
        "it within 60 seconds. Rotate keys quarterly as a baseline practice."
    ),
    "rate-limits": (
        "API rate limiting. The public API allows 120 requests per minute per "
        "key. Exceeding the limit returns HTTP 429 with a Retry-After header. "
        "Batch endpoints have a separate 20 requests per minute limit."
    ),
    "sdk-android": (
        "Android SDK integration. Add the dependency, initialize the client in "
        "Application.onCreate with your publishable key, and call identify "
        "after sign-in. ProGuard rules ship inside the artifact."
    ),
    "sdk-ios": (
        "iOS SDK integration. Install via Swift Package Manager, initialize in "
        "the AppDelegate with your publishable key, and call identify after "
        "sign-in. The SDK requires iOS 15 or later."
    ),
    "sso-setup": (
        "Configuring SAML single sign-on. In the admin console upload your "
        "identity provider metadata, map email and group attributes, and test "
        "with a pilot group before enforcing. SCIM provisioning can be enabled "
        "after single sign-on is verified."
    ),
    "data-export": (
        "Exporting workspace data. Owners can export all workspace data as CSV "
        "or JSON from Settings. Exports are prepared asynchronously and a "
        "download link is emailed within a few hours."
    ),
}

# naive: corpus-blind synonyms. candidates: hand-authored, frozen before scoring.
QUERIES = [
    {
        "id": "q1-error-code",
        "query": "payment failed with error ERR-4032",
        "gold": "pay-err-4032",
        "kind": "exact identifier",
        "naive": ["charge", "declined", "transaction", "billing", "card", "issue"],
        "candidates": ["card", "declined", "avs", "mismatch", "billing",
                        "address", "gateway", "charge", "transaction"],
    },
    {
        "id": "q2-webhook-key",
        "query": "how do I rotate the signing key for webhooks",
        "gold": "webhook-rotate",
        "kind": "lexical-friendly",
        "naive": ["update", "change", "credentials", "security", "api"],
        "candidates": ["secret", "signing", "endpoint", "signature",
                        "revoke", "credentials", "rotate"],
    },
    {
        "id": "q3-slow-app",
        "query": "app is really slow after the last update",
        "gold": "perf-regression",
        "kind": "vocabulary mismatch",
        "naive": ["performance", "speed", "fix", "issue", "problem", "lag"],
        "candidates": ["latency", "degradation", "regression", "p95",
                        "response", "slow", "performance", "rollout", "queries"],
    },
    {
        "id": "q4-pod-restart",
        "query": "pod keeps restarting in production",
        "gold": "crashloop",
        "kind": "vocabulary mismatch",
        "naive": ["kubernetes", "failure", "deployment", "service", "container"],
        "candidates": ["crashloopbackoff", "container", "kubectl", "liveness",
                        "oomkilled", "backoff", "crash", "kubernetes"],
    },
    {
        "id": "q5-data-deletion",
        "query": "customer wants their data deleted",
        "gold": "gdpr-erasure",
        "kind": "vocabulary mismatch",
        "naive": ["remove", "account", "information", "user", "delete"],
        "candidates": ["gdpr", "erasure", "privacy", "purge", "personal",
                        "right", "remove", "account"],
    },
    {
        "id": "q6-pg-upgrade",
        "query": "upgrade postgres major version without downtime",
        "gold": "pg-upgrade",
        "kind": "lexical-friendly",
        "naive": ["database", "migration", "update", "maintenance"],
        "candidates": ["postgresql", "logical", "replication", "replica",
                        "cutover", "primary", "migration", "database"],
    },
    {
        "id": "q7-sso-plan",
        "query": "which plan includes SSO",
        "gold": "pricing-tiers",
        "kind": "vocabulary mismatch",
        "naive": ["pricing", "subscription", "features", "tier", "cost"],
        "candidates": ["saml", "single", "sign-on", "enterprise", "tier",
                        "scim", "pricing", "plan"],
    },
    {
        "id": "q8-version",
        "query": "what changed in v2.14.0",
        "gold": "changelog-2-14",
        "kind": "exact identifier",
        "naive": ["release", "notes", "version", "update", "new", "features"],
        "candidates": ["release", "notes", "permissions", "roles",
                        "dark", "mode", "version"],
    },
    {
        "id": "q9-2fa-reset",
        "query": "reset 2FA for a locked out user",
        "gold": "twofa-reset",
        "kind": "expansion trap",
        "naive": ["account", "login", "security", "access", "password",
                   "authentication", "locked"],
        "candidates": ["two-factor", "totp", "recovery", "re-enroll",
                        "authentication", "account", "login", "security",
                        "password", "device"],
    },
    {
        "id": "q10-redis-timeout",
        "query": "timeout connecting to redis from the worker",
        "gold": "redis-pool",
        "kind": "lexical-friendly",
        "naive": ["connection", "error", "cache", "network", "service"],
        "candidates": ["pool", "connection", "exhaustion", "max_connections",
                        "socket_timeout", "concurrency", "cache"],
    },
    {
        "id": "q11-vat",
        "query": "invoice shows wrong VAT for German customers",
        "gold": "vat-germany",
        "kind": "exact identifier",
        "naive": ["tax", "billing", "invoice", "error", "europe"],
        "candidates": ["vat-1203", "tax", "rate", "19%", "de", "regenerated",
                        "invoices", "germany"],
    },
    {
        "id": "q12-terraform",
        "query": "deploy is stuck at terraform apply",
        "gold": "terraform-lock",
        "kind": "vocabulary mismatch",
        "naive": ["deployment", "infrastructure", "pipeline", "failed", "ci"],
        "candidates": ["state", "lock", "force-unlock", "dynamodb",
                        "hangs", "infrastructure", "pipeline"],
    },
]


## 2. Index, document frequencies, and the five systems

The tokenizer keeps joined identifiers (`err-4032`, `v2.14.0`, `max_connections`) as single tokens. That matters for error codes and version strings.

The corpus-aware system implements two SIRA query-side mechanics:
- **DF filter**: keep a proposed term only if `0 < DF ≤ τ·N` (it exists in the corpus and is rare enough to carry IDF weight)
- **Weighted combination**: `score = BM25(q_orig) + w · BM25(q_exp)` so expansion terms can never drown out the original query

In [ ]:
import re

import numpy as np
from rank_bm25 import BM25Okapi


TOKEN_RE = re.compile(r"[a-z0-9]+(?:[-_.+][a-z0-9]+)*")

DF_TAU = 0.1          # expansion terms must appear in <= 10% of docs
EXPANSION_WEIGHT = 0.5  # w in score = BM25(q_orig) + w * BM25(q_exp)
RRF_K = 60


def tokenize(text: str) -> list[str]:
    """Lowercase tokens; keeps joined identifiers like err-4032, v2.14.0."""
    return TOKEN_RE.findall(text.lower())


DOC_IDS = list(DOCS.keys())
DOC_TOKENS = [tokenize(DOCS[d]) for d in DOC_IDS]
N_DOCS = len(DOC_IDS)

BM25_INDEX = BM25Okapi(DOC_TOKENS)

DF = {}
for toks in DOC_TOKENS:
    for t in set(toks):
        DF[t] = DF.get(t, 0) + 1


def bm25_scores(tokens: list[str]) -> np.ndarray:
    return np.asarray(BM25_INDEX.get_scores(tokens))


def ranks_from_scores(scores: np.ndarray) -> dict[str, int]:
    """doc id -> 1-based rank (stable for ties)."""
    order = np.argsort(-scores, kind="stable")
    return {DOC_IDS[i]: r + 1 for r, i in enumerate(order)}


def run_bm25(q: dict) -> np.ndarray:
    return bm25_scores(tokenize(q["query"]))


_DENSE = None
_DOC_EMB = None


def _dense_model():
    global _DENSE, _DOC_EMB
    if _DENSE is None:
        from sentence_transformers import SentenceTransformer
        _DENSE = SentenceTransformer("all-MiniLM-L6-v2")
        _DOC_EMB = _DENSE.encode(
            [DOCS[d] for d in DOC_IDS], normalize_embeddings=True
        )
    return _DENSE, _DOC_EMB


def run_dense(q: dict) -> np.ndarray:
    model, doc_emb = _dense_model()
    q_emb = model.encode([q["query"]], normalize_embeddings=True)
    return (doc_emb @ q_emb.T).ravel()


def run_naive_expansion(q: dict) -> np.ndarray:
    """Append corpus-blind synonyms to the query, single BM25 call."""
    expanded = tokenize(q["query"]) + [t for s in q["naive"] for t in tokenize(s)]
    return bm25_scores(expanded)


def filter_candidates(candidates: list[str]) -> tuple[list[str], list[dict]]:
    """SIRA-style DF filter: keep terms with 0 < DF <= tau * N."""
    kept, audit = [], []
    max_df = DF_TAU * N_DOCS
    for term in candidates:
        toks = tokenize(term)
        for t in toks:
            df = DF.get(t, 0)
            if df == 0:
                verdict = "dropped (not in corpus)"
            elif df > max_df:
                verdict = "dropped (too common)"
            else:
                verdict = "kept"
                kept.append(t)
            audit.append({"term": t, "df": df, "verdict": verdict})
    return kept, audit


def run_sira_expansion(q: dict) -> np.ndarray:
    kept, _ = filter_candidates(q["candidates"])
    score = bm25_scores(tokenize(q["query"]))
    if kept:
        score = score + EXPANSION_WEIGHT * bm25_scores(kept)
    return score


def run_hybrid(q: dict) -> np.ndarray:
    """Reciprocal rank fusion of vanilla BM25 and dense."""
    fused = np.zeros(N_DOCS)
    for scores in (run_bm25(q), run_dense(q)):
        order = np.argsort(-scores, kind="stable")
        for rank, i in enumerate(order):
            fused[i] += 1.0 / (RRF_K + rank + 1)
    return fused


SYSTEMS = {
    "BM25": run_bm25,
    "Dense (MiniLM)": run_dense,
    "Naive expansion": run_naive_expansion,
    "Corpus-aware expansion": run_sira_expansion,
    "Hybrid (RRF)": run_hybrid,
}


def evaluate() -> dict:
    per_query = []
    for q in QUERIES:
        row = {"id": q["id"], "query": q["query"], "kind": q["kind"],
               "gold": q["gold"], "ranks": {}}
        for name, fn in SYSTEMS.items():
            ranks = ranks_from_scores(fn(q))
            row["ranks"][name] = ranks[q["gold"]]
        per_query.append(row)

    summary = {}
    for name in SYSTEMS:
        gold_ranks = np.array([r["ranks"][name] for r in per_query])
        summary[name] = {
            "MRR": float(np.mean(1.0 / gold_ranks)),
            "Recall@1": float(np.mean(gold_ranks == 1)),
            "Recall@3": float(np.mean(gold_ranks <= 3)),
        }
    return {"per_query": per_query, "summary": summary}


## 3. Evaluate: MRR, Recall@1, Recall@3, and per-query gold ranks

In [ ]:
import pandas as pd

results = evaluate()

summary = pd.DataFrame(results["summary"]).T.round(3)
display(summary)

ranks = pd.DataFrame(
    [{"query": r["id"], "kind": r["kind"], **r["ranks"]}
     for r in results["per_query"]]
).set_index("query")
display(ranks)


## 4. Figures

Seven figures: the compiler-vs-loop diagram (dashed box = full SIRA only), the metric summary, the per-query rank heatmap, the DF-filter audit for the expansion-trap query, the dense identifier-blur examples, the BM25 term-weight heatmap for the compiled paraphrase query, and the hand-authored vs Sonnet 4.6 comparison.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch


FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

INK = "#1f2430"
BLUE = "#3567b3"
ORANGE = "#e07b39"
GREEN = "#2e8b57"
RED = "#c0392b"
GREY = "#9aa3ad"

RESULTS = evaluate()
SYSTEM_NAMES = list(RESULTS["summary"].keys())


def _box(ax, x, y, w, h, text, fc="#eef2f8", ec=BLUE, fs=10, weight="normal"):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.012",
                                fc=fc, ec=ec, lw=1.4))
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center",
            fontsize=fs, color=INK, weight=weight)


def _arrow(ax, xy_from, xy_to, color=INK, style="-|>", conn="arc3,rad=0.0"):
    ax.add_patch(FancyArrowPatch(xy_from, xy_to, arrowstyle=style,
                                 mutation_scale=14, color=color, lw=1.6,
                                 connectionstyle=conn))


def fig_diagram():
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
    for ax in axes:
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.axis("off")

    # Left: agentic loop
    ax = axes[0]
    ax.set_title("Agentic retrieval: the LLM searches more",
                 fontsize=12.5, weight="bold", color=INK, pad=14)
    _box(ax, 0.05, 0.72, 0.28, 0.16, "Query", fc="#fdf3ec", ec=ORANGE)
    _box(ax, 0.36, 0.42, 0.30, 0.18, "LLM agent\n(plan / reflect)", weight="bold")
    _box(ax, 0.05, 0.12, 0.28, 0.16, "Search tool")
    _box(ax, 0.70, 0.12, 0.26, 0.16, "Read / open\ndocs")
    _box(ax, 0.70, 0.72, 0.26, 0.16, "Answer", fc="#edf6ef", ec=GREEN)
    _arrow(ax, (0.26, 0.72), (0.42, 0.60))
    _arrow(ax, (0.42, 0.44), (0.24, 0.28), conn="arc3,rad=0.15")
    _arrow(ax, (0.30, 0.20), (0.44, 0.42), conn="arc3,rad=0.15")
    _arrow(ax, (0.60, 0.44), (0.78, 0.28), conn="arc3,rad=-0.15")
    _arrow(ax, (0.74, 0.24), (0.58, 0.42), conn="arc3,rad=-0.15")
    _arrow(ax, (0.62, 0.56), (0.74, 0.71))
    ax.text(0.51, 0.10, "iterate 3–10×\n(latency & cost multiply)",
            ha="center", fontsize=9.5, color=RED, style="italic")

    # Right: query-side compiler (this experiment)
    ax = axes[1]
    ax.set_title("Query compiler (this experiment): search once, but better",
                 fontsize=12.5, weight="bold", color=INK, pad=14)
    _box(ax, 0.03, 0.72, 0.22, 0.16, "Query", fc="#fdf3ec", ec=ORANGE)
    _box(ax, 0.30, 0.68, 0.34, 0.24,
         "Predict evidence terms\n(hand-authored or LLM)", weight="bold")
    _box(ax, 0.30, 0.36, 0.34, 0.18, "DF filter\n(corpus statistics)")
    _box(ax, 0.30, 0.06, 0.34, 0.18, "One weighted\nBM25 call")
    _box(ax, 0.72, 0.06, 0.24, 0.18, "Answer", fc="#edf6ef", ec=GREEN)
    # Full SIRA enriches docs offline; not part of this experiment (dashed, no arrow).
    _box(ax, 0.03, 0.36, 0.22, 0.18, "Docs enriched\noffline (LLM)\n[full SIRA only]",
         fc="#f4f0fa", ec="#7d5ba6", fs=8.5, weight="normal")
    ax.add_patch(FancyBboxPatch((0.02, 0.34), 0.24, 0.22, boxstyle="round,pad=0.012",
                                fc="none", ec="#7d5ba6", lw=1.2, ls="--"))
    _arrow(ax, (0.25, 0.80), (0.32, 0.80))
    _arrow(ax, (0.47, 0.68), (0.47, 0.55))
    _arrow(ax, (0.47, 0.36), (0.47, 0.25))
    _arrow(ax, (0.64, 0.15), (0.73, 0.15))
    ax.text(0.67, 0.30, "drop absent &\nlow-IDF terms", ha="left",
            fontsize=8.5, color=GREY, style="italic")

    fig.tight_layout()
    fig.savefig(FIG_DIR / "fig1_compiler_vs_loop.png", dpi=200,
                bbox_inches="tight")
    plt.show()


def fig_summary():
    metrics = ["MRR", "Recall@1", "Recall@3"]
    colors = [GREY, BLUE, RED, GREEN, ORANGE]
    x = np.arange(len(metrics))
    width = 0.16

    fig, ax = plt.subplots(figsize=(9, 4.4))
    for i, name in enumerate(SYSTEM_NAMES):
        vals = [RESULTS["summary"][name][m] for m in metrics]
        bars = ax.bar(x + (i - 2) * width, vals, width, label=name,
                      color=colors[i], edgecolor="white")
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width() / 2, v + 0.015, f"{v:.2f}",
                    ha="center", fontsize=8, color=INK)
    ax.set_xticks(x)
    ax.set_xticklabels(metrics, fontsize=11.5)
    ax.set_ylim(0, 1.12)
    ax.set_ylabel("score")
    ax.set_title("Five retrieval systems, 12 queries across 4 regimes, "
                 "36-doc toy corpus", fontsize=12.5, weight="bold", color=INK)
    ax.legend(frameon=False, fontsize=9, ncol=3, loc="lower center",
              bbox_to_anchor=(0.5, -0.32))
    fig.savefig(FIG_DIR / "fig2_summary.png", dpi=200, bbox_inches="tight")
    plt.show()


def fig_rank_heatmap():
    per_q = RESULTS["per_query"]
    ranks = np.array([[r["ranks"][s] for s in SYSTEM_NAMES] for r in per_q])
    # color by capped rank: 1 good (green) -> bad (red)
    capped = np.minimum(ranks, 6)

    fig, ax = plt.subplots(figsize=(8.5, 5.6))
    im = ax.imshow(capped, cmap="RdYlGn_r", vmin=1, vmax=6, aspect="auto")
    for i in range(ranks.shape[0]):
        for j in range(ranks.shape[1]):
            ax.text(j, i, str(ranks[i, j]), ha="center", va="center",
                    fontsize=10.5, weight="bold",
                    color=INK if capped[i, j] < 4 else "white")
    ax.set_xticks(range(len(SYSTEM_NAMES)))
    ax.set_xticklabels([s.replace(" expansion", "\nexpansion")
                        .replace(" (", "\n(") for s in SYSTEM_NAMES],
                       fontsize=9.5)
    labels = [f"{r['id'].split('-', 1)[1]}  ·  {r['kind']}" for r in per_q]
    ax.set_yticks(range(len(per_q)))
    ax.set_yticklabels(labels, fontsize=9.5)
    ax.set_title("Rank of the gold document (1 = retrieved first)",
                 fontsize=12.5, weight="bold", color=INK, pad=12)
    fig.colorbar(im, ax=ax, shrink=0.6, label="rank (capped at 6)")
    fig.savefig(FIG_DIR / "fig3_rank_heatmap.png", dpi=200,
                bbox_inches="tight")
    plt.show()


def fig_df_filter():
    q = next(x for x in QUERIES if x["id"] == "q9-2fa-reset")
    _, audit = filter_candidates(q["candidates"])
    terms = [a["term"] for a in audit]
    dfs = [a["df"] for a in audit]
    kept = [a["verdict"] == "kept" for a in audit]
    colors = [GREEN if k else RED for k in kept]
    cutoff = DF_TAU * N_DOCS

    fig, ax = plt.subplots(figsize=(9, 4.6))
    y = np.arange(len(terms))
    ax.barh(y, dfs, color=colors, edgecolor="white", height=0.62)
    for yi, (df, k, a) in enumerate(zip(dfs, kept, audit)):
        note = "" if k else ("  ✗ " + a["verdict"].split("(")[1].rstrip(")"))
        ax.text(df + 0.08, yi, f"{df}{note}", va="center", fontsize=9.5,
                color=INK)
    ax.axvline(cutoff, color=INK, ls="--", lw=1.2)
    ax.text(cutoff + 0.05, len(terms) - 0.4,
            f"DF cutoff = {DF_TAU:.0%} of corpus", fontsize=9, color=INK)
    ax.set_yticks(y)
    ax.set_yticklabels(terms, fontsize=10.5, family="monospace")
    ax.invert_yaxis()
    ax.set_xlabel("document frequency (of 36 docs)")
    ax.set_title('Compiling the query  "reset 2FA for a locked out user"\n'
                 "candidate terms (hand-authored), validated against corpus statistics",
                 fontsize=12, weight="bold", color=INK)
    handles = [mpatches.Patch(color=GREEN, label="kept"),
               mpatches.Patch(color=RED, label="dropped")]
    ax.legend(handles=handles, frameon=False, loc="lower right")
    fig.savefig(FIG_DIR / "fig4_df_filter.png", dpi=200, bbox_inches="tight")
    plt.show()


def fig_dense_failure():
    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))

    cases = [
        ("q1-error-code", ["pay-err-4032", "pay-err-4031", "pay-general",
                           "pay-retry-policy"],
         '"payment failed with error ERR-4032"'),
        ("q8-version", ["perf-regression", "changelog-2-14", "changelog-2-15",
                        "changelog-2-13"],
         '"what changed in v2.14.0"'),
    ]
    for ax, (qid, doc_ids, title) in zip(axes, cases):
        q = next(x for x in QUERIES if x["id"] == qid)
        dense = run_dense(q)
        bm25 = run_bm25(q)
        bm25 = bm25 / (bm25.max() or 1)          # normalise for display
        idx = [DOC_IDS.index(d) for d in doc_ids]
        y = np.arange(len(doc_ids))
        ax.barh(y - 0.18, [dense[i] for i in idx], height=0.34,
                color=BLUE, label="dense cosine")
        ax.barh(y + 0.18, [bm25[i] for i in idx], height=0.34,
                color=ORANGE, label="BM25 (normalized)")
        labels = [(d + "  ★ gold" if d == q["gold"] else d) for d in doc_ids]
        ax.set_yticks(y)
        ax.set_yticklabels(labels, fontsize=10, family="monospace")
        ax.invert_yaxis()
        ax.set_title(title, fontsize=11.5, weight="bold", color=INK)
        ax.set_xlim(0, 1.05)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, frameon=False, fontsize=10, ncol=2,
               loc="lower center", bbox_to_anchor=(0.5, -0.06))
    fig.suptitle("Where embeddings blur: sibling identifiers and nearby "
                 "release context", fontsize=12.5, weight="bold", color=INK, y=1.08)
    fig.text(0.5, 0.99, "BM25 normalized to max = 1; cosine shown raw. "
             "Compare rank order within each method, not magnitude across "
             "methods.", ha="center", fontsize=9.5, color=GREY,
             style="italic")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "fig5_dense_failure.png", dpi=200,
                bbox_inches="tight")
    plt.show()


STOPWORDS = {"is", "the", "after", "a", "an", "of", "to", "in", "for", "on",
             "with", "at", "really", "last", "and", "or", "my", "do", "how"}


def fig_term_heatmap():
    q = next(x for x in QUERIES if x["id"] == "q3-slow-app")
    orig_terms = [t for t in tokenize(q["query"])
                  if DF.get(t, 0) > 0 and t not in STOPWORDS]
    kept, _ = filter_candidates(q["candidates"])
    exp_terms = [t for t in dict.fromkeys(kept) if t not in orig_terms]
    terms = orig_terms + exp_terms

    doc_ids = ["perf-regression", "changelog-2-14", "pg-tuning",
               "pay-retry-policy", "redis-cache"]
    idx = [DOC_IDS.index(d) for d in doc_ids]
    mat = np.array([[bm25_scores([t])[i] for i in idx] for t in terms])

    fig, ax = plt.subplots(figsize=(8.6, 5.8))
    im = ax.imshow(mat, cmap="Blues", aspect="auto")
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v = mat[i, j]
            if v > 0.01:
                ax.text(j, i, f"{v:.1f}", ha="center", va="center",
                        fontsize=9, color="white" if v > mat.max() * 0.6
                        else INK)
    ax.set_xticks(range(len(doc_ids)))
    ax.set_xticklabels([d + ("\n★ gold" if d == q["gold"] else "")
                        for d in doc_ids], fontsize=9.5)
    ax.set_yticks(range(len(terms)))
    ax.set_yticklabels(terms, fontsize=10, family="monospace")
    for spine in ax.spines.values():
        spine.set_visible(False)
    n_orig = len(orig_terms)
    ax.axhline(n_orig - 0.5, color=ORANGE, lw=2)
    ax.text(len(doc_ids) - 0.4, n_orig - 0.68, "original query terms ↑",
            ha="right", fontsize=9, color=ORANGE, style="italic")
    ax.text(len(doc_ids) - 0.4, n_orig - 0.28, "compiled expansion terms ↓",
            ha="right", fontsize=9, color=GREEN, style="italic")
    ax.set_title('BM25 term-weight heatmap: "app is really slow after the '
                 'last update"\nper-term score contribution by document '
                 "(stopword and out-of-vocabulary rows omitted)",
                 fontsize=11.5, weight="bold", color=INK, pad=12)
    ax.set_xlabel("omitted rows: stopwords (near-zero IDF) and query terms "
                  "absent from the corpus, e.g. 'app', 'update' (zero "
                  "contribution everywhere)", fontsize=8.5,
                  color=GREY, style="italic")
    fig.colorbar(im, ax=ax, shrink=0.7, label="BM25 contribution")
    fig.savefig(FIG_DIR / "fig6_term_heatmap.png", dpi=200,
                bbox_inches="tight")
    plt.show()


def fig_llm_comparison():
    """Hand-authored vs frozen Sonnet 4.6 corpus-aware expansion."""
    cmp_path = FIG_DIR.parent / "results_llm_comparison.json"
    if not cmp_path.exists():
        return
    cmp = json.loads(cmp_path.read_text())
    hand = cmp["hand_authored"]
    llm = cmp["llm_generated"]
    metrics = ["MRR", "Recall@1", "Recall@3"]
    labels = ["Hand-authored\n(upper bound)",
              "Claude Sonnet 4.6\n(frozen run)"]
    colors = [GREEN, BLUE]

    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    x = np.arange(len(metrics))
    width = 0.34
    for i, (label, data, color) in enumerate(zip(labels, [hand, llm], colors)):
        vals = [data[m] for m in metrics]
        offset = (i - 0.5) * width
        bars = ax.bar(x + offset, vals, width, label=label, color=color,
                      edgecolor="white")
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width() / 2, v + 0.015, f"{v:.2f}",
                    ha="center", fontsize=9, color=INK)
    ax.set_xticks(x)
    ax.set_xticklabels(metrics, fontsize=11.5)
    ax.set_ylim(0, 1.12)
    ax.set_ylabel("score")
    ax.set_title("Corpus-aware expansion: hand-authored vs real LLM candidates",
                 fontsize=12, weight="bold", color=INK)
    ax.legend(frameon=False, fontsize=9, loc="upper center",
              bbox_to_anchor=(0.5, -0.12), ncol=2)
    ax.text(0.5, -0.30, "Same DF filter and weighted BM25; only the candidate "
            "source differs. Sonnet misses 1/12 queries (paraphrase).",
            transform=ax.transAxes, fontsize=8.5, color=GREY, style="italic",
            ha="center")
    fig.savefig(FIG_DIR / "fig7_llm_comparison.png", dpi=200,
                bbox_inches="tight")
    plt.show()

fig_diagram()
fig_summary()
fig_rank_heatmap()
fig_df_filter()
fig_dense_failure()
fig_term_heatmap()
fig_llm_comparison()

## 5. Optional: live LLM expansion

Swap the frozen candidate lists for live model output. Results vary run to run.

In [ ]:
# OPTIONAL: replace the frozen expansion lists with live LLM output.
# Set ANTHROPIC_API_KEY, run this cell, then rerun the evaluation cells.
#
# import os, anthropic
#
# client = anthropic.Anthropic()  # uses ANTHROPIC_API_KEY
#
# PROMPT = (
#     "You are helping a lexical (BM25) search engine. For the user query "
#     "below, list 8-12 single words or short phrases likely to appear "
#     "verbatim in the one document that answers it. Prefer specific, "
#     "discriminative vocabulary (error codes, exact feature names, "
#     "domain terms) over generic synonyms. Return one term per line.\n\n"
#     "Query: {query}"
# )
#
# for q in QUERIES:
#     msg = client.messages.create(
#         model="claude-sonnet-4-6",
#         max_tokens=300,
#         messages=[{"role": "user",
#                    "content": PROMPT.format(query=q["query"])}],
#     )
#     q["candidates"] = [t.strip() for t in msg.content[0].text.splitlines()
#                        if t.strip()]


## Takeaways

- Vanilla BM25 dies on paraphrase (gold rank 14 on the "slow app" query); dense dies on identifiers (ranks the wrong error code and a performance retro above the gold changelog).
- Naive expansion is worse than no expansion (MRR 0.80 vs 0.84).
- Hand-authored candidates score 1.000; frozen Sonnet 4.6 candidates score 0.922 MRR (11/12 at rank 1). The miss is the paraphrase query: the model guessed `degraded`/`regression`, the corpus says `latency`/`p95`/`N+1`.
- A 36-doc corpus is a teaching instrument, not a benchmark. For real numbers see SIRA's BEIR results ([arXiv:2605.06647](https://arxiv.org/abs/2605.06647)) and Pi-Serini ([arXiv:2605.10848](https://arxiv.org/abs/2605.10848)).